In [ ]:
import torch
import numpy as np
from scipy.stats import pearsonr
from sklearn import linear_model
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist,squareform
from torch.utils.data import Dataset
import torch.nn as nn
import re
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from collections import defaultdict
from tqdm import tqdm
import torch.nn.functional as F
import random
import copy
from pathlib import Path

## ID Estimation Algorithms

Core functions to estimate Intrinsic Dimension (ID) from feature activations.

* **`twonn_id`**
* **`mle_id`**
* **`repeat_compute`**: A stability wrapper that runs an estimator multiple times on random data subsets and returns the average ID.

> **Note:** Distance computations are GPU-accelerated and batched to prevent memory overflow.

In [ ]:
def twonn_id(X, device='cuda', batch=512, fraction=0.9, verbose=False, subsample=None):

    # --- 1. Subsampling (Optional) ---
    if subsample is not None:
        N = X.shape[0]
        n_sub = int(N * subsample)
        perm = torch.randperm(N)[:n_sub]
        X = X[perm]

    # --- 2. Data Prep ---
    if not isinstance(X, torch.Tensor):
        X = torch.tensor(X)

    X = X.double().to(device)

    N = X.shape[0]

    r1_list = []
    r2_list = []

    # --- 3. Compute Nearest Neighbors (Batchwise) ---
    with torch.no_grad():
        for i in range(0, N, batch):

            Xi = X[i:i+batch]
            Di = torch.cdist(Xi, X)
            vals, _ = torch.topk(Di, k=3, dim=1, largest=False)
            vals = vals.cpu().numpy()

            r1_list.append(vals[:, 1])
            r2_list.append(vals[:, 2])

    r1 = np.concatenate(r1_list)
    r2 = np.concatenate(r2_list)

    # --- 4. Clean Data ---
    mask_nonzero = r1 > 0
    mask_distinct = r2 > r1

    good = mask_nonzero & mask_distinct

    r1 = r1[good]
    r2 = r2[good]
    N_good = r1.shape[0]

    if verbose:
        print(f"[TwoNN] N={N}, Valid={N_good}")

    if N_good < 5:
        return 1.0

    # --- 5. TwoNN Statistics ---
    mu = np.sort(r2 / r1)

    Femp = np.arange(1, N_good + 1, dtype=np.float64) / N_good

    x = np.log(mu[:-2])
    y = -np.log(1 - Femp[:-2])

    npoints = int(np.floor(N_good * fraction))

    if npoints < 2:
        return 1.0

    regr = linear_model.LinearRegression(fit_intercept=False)
    regr.fit(x[:npoints, np.newaxis], y[:npoints, np.newaxis])

    d_hat = regr.coef_[0][0]

    return d_hat

def mle_id(X,device,k=10,batch=128,subsample=None,big_value=1000):

    if subsample is not None:
        N = X.shape[0]
        n_sub = int(N * subsample)
        perm = torch.randperm(N)[:n_sub]
        X = X[perm]

    X = X.to(device)
    N = X.size(0)

    out = []
    with torch.inference_mode():
        for i in range(0, N, batch):
            D = torch.cdist(X[i:i+batch], X)
            vals, _ = torch.topk(D, k=k+1, dim=1, largest=False)
            rk = vals[:, -1]
            logs = torch.log(rk.unsqueeze(1) / vals[:, 1:k])
            mi = (k - 1) / logs.sum(dim=1)
            out.append(mi.cpu())

    out = torch.cat(out)

    # --- Robust Averaging ---
    out = out[torch.isfinite(out)]
    if len(out) == 0:
        return float(big_value)

    denom = (1.0 / out).mean().item()

    if denom == 0 or not np.isfinite(denom):
        return float(big_value)

    id_est = 1.0 / denom

    if not np.isfinite(id_est):
        return float(big_value)

    return float(id_est)
    #return 1.0 / (1.0 / out).mean().item()



def repeat_compute(X,estimator,nres=3,fraction=0.9):

    """
    Bootstrap aggregation (Bagging) wrapper to improve stability.
    Runs the estimator multiple times on random subsets and averages results.
    """

    ID = []
    n = int(np.round(X.shape[0]*fraction))
    for i in range(nres):
        perm = np.random.permutation(X.shape[0])[:n]
        X_s = X[perm]
        d_hat= estimator(X_s)
        ID.append(d_hat)

    mean = np.mean(ID).item()
    #error = np.std(ID).item()
    return mean

# Function: `compute_id_dynamics_across_models`

This is the core analysis engine of the experiment. It orchestrates the extraction of hidden representations from deep neural networks and computes their geometric properties (Intrinsic Dimension and PCA-based dimensionality).

### Key Functionalities

1.  **Flexible Layer Profiling (`depth_fns`)**:
    * It does not blindly hook every layer. Instead, it relies on `depth_fns` to define the specific "checkpoints" (modules) within a model architecture that constitute its depth.
    * This allows for consistent comparison between disparate architectures (e.g., matching a ResNet Block to an MLP Layer).

2.  **Ensemble Model Support (TabM Specifics)**:
    * **Automatic 3D Handling:** It detects if a layer outputs a 3D tensor `(Batch, k, D)`, which is characteristic of Ensemble models like **TabM**.
    * **Flattening Strategy:** It automatically flattens these outputs to `(Batch * k, D)` before ID computation. This ensures that the Intrinsic Dimension calculation considers the geometric contribution of all ensemble members as independent samples, rather than averaging them out. This choice was motivated by our preliminary experiments, which indicated that averaging leads to a collapse of the intrinsic dimension (often yielding invalid estimates), as it smooths out the distinct feature variations learned by different ensemble heads, effectively destroying the geometric complexity that ID aims to measure.

3.  **Efficiency Modes**:
    * **`only_last_hidden_layer`**: A fast-track mode that restricts computation to the final representation layer. This is crucial for tracking dynamics during training (e.g., per-batch logging) without the heavy overhead of computing ID for the entire network.
    * **`pca_dim`**: A toggle to enable/disable linear dimensionality analysis (PCA). Disabling this speeds up the process when only the non-linear Intrinsic Dimension (TwoNN/MLE) is needed.

4.  **Mixed-Input Data Support**:
    * The function detects whether the dataloader yields standard inputs `(x, y)` or tabular-specific separated inputs `(x_num, x_cat, y)`. This ensures compatibility with both standard Vision datasets and complex Tabular datasets.

---

### Parameters
* **`models`** (`dict`): Dictionary of PyTorch models to evaluate.
* **`estimators`** (`dict`): Dictionary of ID estimation functions (e.g., TwoNN, MLE).
* **`dataloader`** (`DataLoader`): Source of data for generating activations.
* **`depth_fns`** (`dict`, optional): Custom logic to extract `(modules, names, depths)` from specific model types.
* **`pca_dim`** (`bool`): Whether to compute PCA effective dimension (requires `sklearn`).
* **`only_last_hidden_layer`** (`bool`): Optimization flag to measure only the penultimate layer.

### Returns
A nested dictionary containing the geometric profile for each model:
```python
{
    "ModelName": {
        "depths": [0, 1, 2, ...],       # The relative depth indices
        "TwoNN":  [4.5, 12.1, ...],     # The Intrinsic Dimension per layer
        "pca_dim": [10, 50, ...],       # (If enabled) PCA effective dimension
        "embdims": [512, 512, ...]      # The physical embedding dimension (D)
        "depth_names": ["input", "", ...]      #The name per layer
    }
}

In [ ]:

th = 0.9

def get_pca_dim(x, th):
    cs = np.cumsum(x)
    indices = np.argwhere(cs > th)

    if indices.size > 0:
        return indices[0][0].item()
    else:
        return 0


def compute_id_dynamics_across_models(models, estimators, dataloader, depth_fns,
                                      device, pca_dim=False,
                                      only_last_hidden_layer=False,
                                      show_progress=True,
                                      ):

    if show_progress:
        iterator = tqdm(models.items(), desc="Models")
    else:
        iterator = models.items()

    id_all_model = {}
    for model_name, model in iterator:

        model_dict = {}
        model = model.to(device)
        model = model.eval()

        # 1. Determine layers to hook (modules) and their depths

        modules, names, depths = depth_fns[model_name](model)
        id_all_model[model_name] = {}
        id_all_model[model_name]['depths'] = depths
        id_all_model[model_name]['depth_names'] = names

        has_input = (len(modules) > 0 and isinstance(modules[0], str) and modules[0] == "input")

        if has_input:
            modules = modules[1:]

        # 2. Filter for only last hidden layer if requested
        if only_last_hidden_layer:
            modules = modules[-2:-1]
            names = names[-2:-1]
            depths = depths[-2:-1]


        # 3. Register hooks for all checkpoints (single forward pass per batch)
        # Unlike a layer-by-layer approach (one forward per layer), we register hooks once and
        # obtain all layer activations in a single forward pass per batch.
        n_layers = len(modules)
        activations_all_layer = [[] for _ in range(n_layers)]

        def make_hook(layer_idx):
            def hook(module, input, output):
                if output.ndim == 3:
                    # TabM-style ensemble output: treat k members as independent samples
                    B, k, D = output.shape
                    output = output.reshape(B * k, D)
                output = output.reshape(output.shape[0], -1)
                activations_all_layer[layer_idx].append(
                    output.detach().cpu()
                )
            return hook

        handles = []
        for i, module in enumerate(modules):
            h = module.register_forward_hook(make_hook(i))
            handles.append(h)


        # 4. One pass over dataloader to collect activations
        activations_input = []
        with torch.no_grad():
            for k, batch in enumerate(dataloader, 0):
                if len(batch) == 3:
                    x_num, x_cat, y = batch
                    inputs = (x_num.to(device), x_cat.to(device))
                else:
                    x, y = batch
                    inputs = x.to(device)
                if has_input and not only_last_hidden_layer:
                    activations_input.append(inputs.reshape(inputs.shape[0], -1).detach().cpu())

                _ = model(inputs)

        for h in handles:
            h.remove()


        # 5. store results
        activations_all_layer = [
            torch.cat(layer_acts, dim=0)
            for layer_acts in activations_all_layer
        ]

        if has_input and not only_last_hidden_layer:
            X_in = torch.cat(activations_input, dim=0)
            activations_all_layer = [X_in] + activations_all_layer

        for estimator_name, estimator_fun in estimators.items():
            ids_all_layer = [estimator_fun(X) for X in activations_all_layer]
            id_all_model[model_name][estimator_name] = ids_all_layer

        if pca_dim:

            pca_all_layer = []
            for X in activations_all_layer:
                scaler = StandardScaler()
                Xn = scaler.fit_transform(X.numpy() if torch.is_tensor(X) else X)
                pca = PCA()
                pca.fit(Xn)
                pca_all_layer.append(get_pca_dim(pca.explained_variance_ratio_, th))

            id_all_model[model_name]['embdims'] = [X.shape[1] for X in activations_all_layer]
            id_all_model[model_name]['pca_dim'] = pca_all_layer


        model.to("cpu")
        torch.cuda.empty_cache()

    return id_all_model


## Core Training & Evaluation Functions

This section defines the low-level functions for executing a single training epoch and evaluating model performance.

### Key Features
* **TabM Support:** Both functions automatically detect if the model output is 3D `(Batch, k, Classes)`.
    * **Training:** It flattens the output to `(Batch * k, Classes)` and repeats targets to compute the loss over all ensemble members.
    * **Inference:** It averages logits across the ensemble dimension `k` (`mean(dim=1)`) to leverage the ensemble prediction for accuracy calculation.
* **ID Logging:** The `train` function includes a hook to periodically compute and log the Intrinsic Dimension of the last hidden layer during training.

### Functions
* **`train`**: Performs one epoch of gradient descent, handling loss computation and periodic logging.
* **`test`**: Evaluates the model on a given dataloader (used for Test/Validation sets).

In [ ]:
def train(global_batch,
          model_dict,
          estimator_dict,
          train_loader,
          test_loader,
          criterion,
          optimizer,
          id_logging_interval,
          device,
          depth_fns=None,
          batch_scheduler=None):

    batch_log = []
    (model_name, model), = model_dict.items()
    (estimator_name, estimator), = estimator_dict.items()
    model.train()
    train_loss = 0
    train_correct = 0
    total = 0
    model.to(device)


    for batch in train_loader:
        # Handle Tabular (3-item) vs Image (2-item) batches
        if len(batch) == 3:
            x_num, x_cat, y = batch
            x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
            inputs = (x_num, x_cat)
        else:
            inputs, y = batch
            inputs, y = inputs.to(device), y.to(device)

        batch_size = y.size(0)
        optimizer.zero_grad()
        outputs = model(inputs)

        # --- TabM Specific Logic (3D Output) ---
        if outputs.ndim == 3:
            B, k, C = outputs.shape
            outputs_flat = outputs.reshape(B * k, C)
            y_flat = y.repeat_interleave(k)
            loss = criterion(outputs_flat, y_flat)
            ensemble_logits = outputs.mean(dim=1)
            preds = ensemble_logits.argmax(dim=1)
        # --- Standard Logic (2D Output) ---
        else:
            loss = criterion(outputs, y)
            preds = outputs.argmax(dim=1)


        loss.backward()
        optimizer.step()

        if batch_scheduler is not None:
            batch_scheduler.step()

        total += batch_size
        train_loss += loss.item() * batch_size

        batch_train_acc = (preds == y).sum().item()
        train_correct += batch_train_acc

        batch_train_acc = batch_train_acc / batch_size

        batch_last_hidden_layer_id = None
        batch_test_acc = None

        # Periodically log ID and Test Accuracy
        if global_batch % id_logging_interval == 0:
            batch_last_hidden_layer_id_dict = compute_id_dynamics_across_models(
                {model_name:model},
                estimator_dict,
                test_loader,
                depth_fns,
                device,
                pca_dim=False,
                only_last_hidden_layer=True,
                show_progress=False)
            batch_last_hidden_layer_id = batch_last_hidden_layer_id_dict[model_name][estimator_name][0]
            _, batch_test_acc = test(model,test_loader,criterion,device)
            model.train()
            model.to(device)

            batch_log.append([global_batch,batch_train_acc,batch_test_acc,batch_last_hidden_layer_id])
        global_batch += 1

    epoch_train_loss = train_loss / total
    epoch_train_acc = train_correct / total
    epoch_test_loss, epoch_test_acc = test(model,test_loader,criterion,device)
    epoch_log = [epoch_train_acc,epoch_test_acc]

    return global_batch, {model_name:model} , batch_log, epoch_log



def test(model,dataloader,criterion,device):


    model.eval()
    test_loss = 0
    test_correct = 0
    total = 0
    model.to(device)

    with torch.no_grad():
        for batch in dataloader:
            if len(batch) == 3:
                x_num, x_cat, y = batch
                x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
                inputs = (x_num, x_cat)
            else:
                inputs, y = batch
                inputs, y = inputs.to(device), y.to(device)

            batch_size = y.size(0)
            outputs = model(inputs)

            if outputs.ndim == 3:

                B, k, C = outputs.shape
                outputs_flat = outputs.reshape(B * k, C)
                y_flat = y.repeat_interleave(k)
                loss = criterion(outputs_flat, y_flat)

                ensemble_logits = outputs.mean(dim=1)
                preds = ensemble_logits.argmax(dim=1)

            else:

                loss = criterion(outputs, y)
                preds = outputs.argmax(dim=1)


            total += batch_size
            test_loss += loss.item() * batch_size

            test_correct += (preds == y).sum().item()


    epoch_test_loss = test_loss / total
    epoch_test_acc = test_correct / total

    return epoch_test_loss, epoch_test_acc




## `train_and_compute_id`

This function manages the full lifecycle of the training experiment.

### Responsibilities
1.  **Baseline Measurement:** Computes the Intrinsic Dimension (ID) of the *untrained* model (Epoch 0).
2.  **Training Loop:** Iterates through epochs, calling the `train` function and updating the model.
3.  **Best Model Tracking:** Monitors test accuracy and saves a copy of the model state that achieves the highest accuracy.
4.  **Full-Spectrum Analysis:**
    * Logs per-batch dynamics (ID & Accuracy).
    * Logs per-epoch dynamics.
    * Computes detailed geometric properties (ID, PCA-dim, Embedding dim) for the **Best Model** found during training.

### Returns
A comprehensive set of logs and the best performing model instance, ready for final visualization.

In [ ]:
def train_and_compute_id(
    epochs,
    model,
    train_loader,
    test_loader,
    estimator,
    criterion,
    optimizer,
    id_logging_interval,
    device,
    batch_scheduler=None,
    epoch_scheduler=None,
    depth_fns=None):



    (model_name, net), = model.items()
    result_per_batch = []
    result_per_epoch = []
    result_0_and_best_epoch = []
    global_batch = 0
    best_acc = 0
    best_model = None
    best_epoch = 0

    # 1. Compute ID for Epoch 0 (Untrained)
    id_dynamics_across_models = compute_id_dynamics_across_models(
        model,estimator,
        test_loader,
        depth_fns,
        device,pca_dim=True,
        show_progress=False)[next(iter(model))]

    result_epoch_0 = [0,test(net,test_loader,criterion,device)[1],
                      id_dynamics_across_models[next(iter(estimator))],
                      id_dynamics_across_models['embdims'],
                      id_dynamics_across_models['pca_dim'],
                      id_dynamics_across_models['depth_names'],
                      ]
    result_per_epoch.append(result_epoch_0[:3] + result_epoch_0[-1:])
    result_0_and_best_epoch.append(result_epoch_0)
    depths = id_dynamics_across_models['depths']

    # 2. Main Training Loop
    for epoch in tqdm(range(1,epochs+1),desc = "Training epochs"):
        global_batch, model, batch_log, epoch_log = train(
            global_batch,
            model,
            estimator,
            train_loader,
            test_loader,
            criterion,
            optimizer,
            id_logging_interval,
            device,
            depth_fns=depth_fns,
            batch_scheduler=batch_scheduler)

        # Track Best Model
        if epoch_log[1] > best_acc:
            best_acc = epoch_log[1]
            best_model = copy.deepcopy(model)
            best_epoch = epoch


        result_per_batch.extend(batch_log)

        # Compute ID for current epoch
        id_dynamics_across_models = compute_id_dynamics_across_models(
                model,
                estimator,
                test_loader,
                depth_fns,
                device,pca_dim=False,
                show_progress=False)[next(iter(model))]

        epoch_id = id_dynamics_across_models[next(iter(estimator))]
        result_per_epoch.append([epoch,epoch_log[1],epoch_id])
        #print(f'epoch{epoch}, train and test acc: {epoch_log}, epoch_id:{epoch_id}')
        if epoch_scheduler is not None:
            epoch_scheduler.step()

    # 3. Final Analysis on Best Model
    id_dynamics_across_models = compute_id_dynamics_across_models(
        best_model,estimator,
        test_loader,
        depth_fns,
        device,pca_dim=True,
        show_progress=False)[next(iter(model))]
    result_0_and_best_epoch.append([best_epoch,best_acc,
                                            id_dynamics_across_models[next(iter(estimator))],
                                            id_dynamics_across_models['embdims'],
                                            id_dynamics_across_models['pca_dim']])
    return depths, result_per_batch, result_per_epoch, result_0_and_best_epoch, best_model

### Model utilities for intrinsic-dimension experiments

This section defines the CNN architectures used throughout the CIFAR-10 experiments (**Fig. 5C**, and **Fig. 9**):

* Custom `VGG class`
A VGG-style network following the standard VGG convolutional configuration, but with the classifier replaced by a single `Linear(512 → 10)` layer.  

* `AlexNet` architecture

    CIFAR-10 AlexNet adapted from the original design: all convolutions use `3×3` kernels, the feature extractor ends with a `128×4×4` map, and the classifier is `Linear(128·4·4 → 1024) → 1024 → num_classes(=10`).

* `ResNet` architectures

    CIFAR-10 ResNets adapted from the standard He et al. implementation:
    initial conv: `3×3` instead of `7×7`

    no initial max-pool

    final pooling adjusted for `32×32` inputs  

    Includes implementations of ResNet-18/34 (BasicBlock) and ResNet-50/101/152 (Bottleneck).


In [ ]:
cfg = {
    'VGG11': [64, 'M', 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M'],
    'VGG13': [64, 64, 'M', 128, 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M'],
    'VGG16': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512, 'M', 512, 512, 512, 'M'],
    'VGG19': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 256, 'M', 512, 512, 512, 512, 'M', 512, 512, 512, 512, 'M'],
}


class VGG(nn.Module):
    def __init__(self, vgg_name):
        super(VGG, self).__init__()
        self.features = self._make_layers(cfg[vgg_name])
        self.classifier = nn.Linear(512, 10)

    def forward(self, x):
        out = self.features(x)
        out = out.view(out.size(0), -1)
        out = self.classifier(out)
        return out

    def _make_layers(self, cfg):
        layers = []
        in_channels = 3
        for x in cfg:
            if x == 'M':
                layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
            else:
                layers += [nn.Conv2d(in_channels, x, kernel_size=3, padding=1),
                           nn.BatchNorm2d(x),
                           nn.ReLU(inplace=True)]
                in_channels = x
        layers += [nn.AvgPool2d(kernel_size=1, stride=1)]
        return nn.Sequential(*layers)



class AlexNet(nn.Module):

    def __init__(self,num_classes=10):
        super(AlexNet,self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 48, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(48, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(128, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(192, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(192, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
      )
        self.classifier = nn.Sequential(
              nn.Dropout(0.5),
              nn.Linear(128 * 4 * 4, 1024),
              nn.ReLU(inplace=True),
              nn.Dropout(0.5),
              nn.Linear(1024, 1024),
              nn.ReLU(inplace=True),
              nn.Linear(1024, num_classes),
        )
    def forward(self,x):

        x = self.features(x)
        x = x.view(x.size(0),-1)
        x = self.classifier(x)
        return x



class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(
            in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_planes, planes, stride=1):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, self.expansion *
                               planes, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(self.expansion*planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet, self).__init__()
        self.in_planes = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.feature_identity = nn.Identity()
        self.linear = nn.Linear(512*block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.feature_identity(out)
        out = self.linear(out)
        return out


def ResNet18():
    return ResNet(BasicBlock, [2, 2, 2, 2])


def ResNet34():
    return ResNet(BasicBlock, [3, 4, 6, 3])


def ResNet50():
    return ResNet(Bottleneck, [3, 4, 6, 3])


def ResNet101():
    return ResNet(Bottleneck, [3, 4, 23, 3])


def ResNet152():
    return ResNet(Bottleneck, [3, 8, 36, 3])



### Layer-enumeration utilities for intrinsic-dimension computation

* Depth extraction utilities (`getDepths`, `getResNetsDepths`, `getDepths_cifar_resnet`)  

    Traverse model layers and record a small set of “checkpoints” (input, pooling / residual blocks, classifier) together with a monotonically increasing depth index.  
      
    These depth indices are later used to plot intrinsic dimension vs. relative depth.

* `ImagesOnlyDataset`

    A simple wrapper around a base dataset that returns only images (no labels), used when computing ID on activations since labels are unnecessary.

In [ ]:
def getDepths(model):
    count = 0
    modules = []
    names = []
    depths = []
    modules.append('input')
    names.append('input')
    depths.append(0)

    for i,module in enumerate(model.features):
        name = module.__class__.__name__
        if 'Conv2d' in name or 'Linear' in name:
            count += 1
        if 'MaxPool2d' in name:
            modules.append(module)
            depths.append(count)
            names.append('MaxPool2d')

    clf = model.classifier
    if isinstance(clf, nn.Sequential):
        classifier_layers = clf
    else:
        classifier_layers = [clf]

    for i,module in enumerate(classifier_layers):
        name = module.__class__.__name__
        if 'Linear' in name:
            modules.append(module)
            count += 1
            depths.append(count + 1)
            names.append('Linear')
    depths = np.array(depths)
    return modules, names, depths




def getLayerDepth(layer):
    count = 0
    for m in layer:
        for c in m.children():
            name = c.__class__.__name__
            if 'Conv' in name:
                count += 1
    return count

def getResNetsDepths(model):
    modules = []
    names = []
    depths = []

    # input
    count = 0
    modules.append('input')
    names.append('input')
    depths.append(count)
    # maxpooling
    count += 1
    modules.append(model.maxpool)
    names.append('maxpool')
    depths.append(count)
    # 1
    count += getLayerDepth(model.layer1)
    modules.append(model.layer1)
    names.append('layer1')
    depths.append(count)
    # 2
    count += getLayerDepth(model.layer2)
    modules.append(model.layer2)
    names.append('layer2')
    depths.append(count)
    # 3
    count += getLayerDepth(model.layer3)
    modules.append(model.layer3)
    names.append('layer3')
    depths.append(count)
    # 4
    count += getLayerDepth(model.layer4)
    modules.append(model.layer4)
    names.append('layer4')
    depths.append(count)
    # average pooling
    count += 1
    modules.append(model.avgpool)
    names.append('avgpool')
    depths.append(count)
    # output
    count += 1
    modules.append(model.fc)
    names.append('fc')
    depths.append(count)
    depths = np.array(depths)
    return modules, names, depths


def getDepths_cifar_resnet(model):
    modules = []
    names = []
    depths = []

    # input
    count = 0
    modules.append("input")
    names.append("input")
    depths.append(count)

    # conv1
    count += 1
    modules.append(model.conv1)
    names.append("conv1")
    depths.append(count)

    # layer1~4
    for layer, lname in [
        (model.layer1, "layer1"),
        (model.layer2, "layer2"),
        (model.layer3, "layer3"),
        (model.layer4, "layer4"),
    ]:
        count += getLayerDepth(layer)
        modules.append(layer)
        names.append(lname)
        depths.append(count)


    count += 1
    modules.append(model.feature_identity)
    names.append("feature_identity")
    depths.append(count)

    count += 1
    modules.append(model.linear)
    names.append("linear")
    depths.append(count)

    depths = np.array(depths)
    return modules, names, depths

class ImagesOnlyDataset(Dataset):
    def __init__(self, base_dataset):
        self.base = base_dataset

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, _ = self.base[idx]
        return img



### Plotting utilities for ID analysis
* Color/marker helpers by model family
* `plot_fig3b`: layer-wise ID vs relative depth for multiple models
* `plot_fig9A/B/C`: ID dynamics over epochs and vs error
* `plot_fig5c`: trained vs untrained ID / PC-ID / ED across depth

In [ ]:
def title_to_filename(title, ext="png"):
    """
    Convert a plot title into a filesystem-safe filename while preserving semantics.

    """
    s = title.lower()
    s = s.replace("&", "and")
    s = re.sub(r"[^\w\s-]", "", s)
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"_+", "_", s)
    s = s.strip("_")

    return f"{s}.{ext}"

def family_color(name: str):
    n = name.lower()
    if "alex" in n:   return "#f2a23a"
    if "vgg" in n and "bn" in n: return "k"
    if "vgg" in n:    return "#1f77b4"
    if "resnet" in n: return "#2ca02c"

    rng = random.Random(n)

    r = rng.randint(0, 200)
    g = rng.randint(0, 200)
    b = rng.randint(0, 200)

    return '#%02X%02X%02X' % (r, g, b)

def model_depth_number(name: str) -> int:
    nums = re.findall(r"\d+", name)
    return int(nums[-1]) if nums else 10


def family_key(name: str):
    n = name.lower()
    if "resnet" in n: return "resnet"
    if "vgg" in n and "bn" in n: return "vgg_bn"
    if "vgg" in n: return "vgg"
    if "alex" in n: return "alexnet"
    return "other"

def build_family_stats(data):
    depths = defaultdict(list)
    for model_name,model_measures in data.items():
        depths[family_key(model_name)].append(model_depth_number(model_name))
    stats = {}
    for k, vals in depths.items():
        dmin, dmax = min(vals), max(vals)
        stats[k] = (dmin, dmax)
    return stats

def marker_size_for_model_family(name, family_stats, min_s=30, max_s=90):
    fam = family_key(name)
    d = model_depth_number(name)
    dmin, dmax = family_stats[fam]
    if dmax == dmin:
        return (min_s + max_s) / 2
    frac = (d - dmin) / (dmax - dmin)
    s = min_s + (max_s - min_s) * frac
    return float(np.clip(s, min_s, max_s))


def plot_fig3b(data,estimator_name,title,annotate=False,show=True,savepath=None):

    """
    Plot layer-wise intrinsic dimension curves for multiple models (reproducing Fig. 3B).

    Args:
        data (dict):
            Output of compute_id_dynamics_across_models().
            For each model:
                - data[model_name]["depths"]: list of layer depth indices
                - data[model_name][estimator_name]: list of ID values (one per layer)
        estimator_name (str): Key selecting which ID estimator to plot ("TwoNN", "MLE", etc.).
        title (str): Plot title (also used to derive a safe filename when saving).

    Note:
        `data[model_name][estimator_name]` **is the list of layer-wise ID values**.
        This function only visualizes precomputed results.
    """

    plt.figure(figsize=(9,6))
    legend_handles = []
    legend_labels  = []

    family_stats = build_family_stats(data)
    for model_name,model_measures in data.items():
        layers = np.asarray(model_measures["depths"], dtype=float)
        ids = np.asarray(model_measures[estimator_name], dtype=float)

        depth_names = model_measures.get("depth_names", None)

        max_layer = max(layers.max(), 1.0)
        x = layers / max_layer
        y = ids


        color = family_color(model_name)
        s = marker_size_for_model_family(model_name,family_stats)


        plt.plot(x, y, "-", lw=2, color=color, alpha=0.9, label=model_name)
        plt.scatter(x, y, s=s, color=color, edgecolors="white", linewidths=1.2, zorder=3)

        dx = -0.05
        dy = 0.05
        if annotate and depth_names is not None:
            for xi, yi, name in zip(x, y, depth_names):
                plt.text(
                    xi + dx, yi + dy,
                    name,
                    color=color,
                    fontsize=7,
                    alpha=0.9
                )

        handle = Line2D([0], [0], color=color, marker='o', linestyle='-',
                        markerfacecolor=color, markeredgecolor='white',
                        linewidth=2, markersize=np.sqrt(s),
                        label=model_name)
        legend_handles.append(handle)
        legend_labels.append(model_name)


    plt.xlabel("relative depth", fontsize=13)
    plt.ylabel("ID", fontsize=13)
    plt.title(title, fontsize=14)

    plt.xlim(-0.02, 1.02)
    #plt.ylim(0, 160)
    plt.xticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])


    uniq = dict(zip(legend_labels, legend_handles))
    plt.legend(uniq.values(), uniq.keys(), ncol=2, frameon=False, fontsize=11,bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()

    if savepath is not None:
        outdir = Path(savepath)
        outdir.mkdir(parents=True, exist_ok=True)
        outfile = outdir / title_to_filename(title)
        plt.savefig(outfile, dpi=300, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close(plt.gcf())






def plot_fig9A(epoch_records, depths, title, annotate=False, show=True, savepath=None):


    """
    Plot the evolution of layer-wise intrinsic dimension across epochs (Fig. 9A reproduction).

    Args:
        epoch_records (list):
            List where each element corresponds to one epoch.
            Each entry has the format:
                [epoch_index, epoch_acc, id_values]
            - epoch_index (int): epoch number
            - epoch_acc (float or None): test accuracy for that epoch
            - id_values (list):
                  Layer-wise intrinsic dimension values, each entry is a scalar.

        depths (array-like):
            Depth index for each layer (e.g., 0, 1, 2, ...); used to compute relative depth.

    Notes:
        - `epoch_records[0]` is interpreted as the UNTRAINED network.
        - For each epoch, only the ID mean is plotted.
        - Color encodes epoch progression (Viridis colormap).
        - X-axis uses relative depth = depth / max(depth).

    This function visualizes precomputed ID dynamics, but does not compute ID.
    """


    num_epochs = len(epoch_records)
    cmap = cm.get_cmap("viridis")
    norm = Normalize(vmin=0, vmax=num_epochs - 1)

    fig, ax = plt.subplots(figsize=(8,6))

    rel_depth = depths / depths[-1]

    epoch0_id = epoch_records[0][2]
    ax.plot(rel_depth, epoch0_id, color="black", linewidth=3, label="UNTRAINED")

    for i in range(1, num_epochs):

        epoch_idx, _, id_values = epoch_records[i]
        color = cmap(norm(i))

        ax.plot(rel_depth, id_values, color=color, linewidth=1.3, alpha=0.9)

        if i <= 3:
            ax.text(rel_depth[1], id_values[1],
                    f"EPOCH {epoch_idx}",
                    color=color, fontsize=8, rotation=25)

        if i == num_epochs - 1:
            ax.text(rel_depth[1], id_values[1],
                    f"EPOCH {epoch_idx}",
                    color=color, fontsize=8, rotation=25)

    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label("epoch")

    ax.set_xlabel("relative depth")
    ax.set_ylabel("ID")
    #ax.set_ylim(0, 80)

    if annotate:
        depth_names = epoch_records[0][-1]
        y_top = ax.get_ylim()[1]

        for i in range(len(depth_names) - 1):
            x_l = rel_depth[i]
            x_r = rel_depth[i + 1]
            x_mid = 0.5 * (x_l + x_r)

            stage_label = depth_names[i + 1]

            ax.text(
                x_mid,
                y_top*1.01,
                stage_label,
                ha="center",
                va="bottom",
                fontsize=8,
                color="black",
                alpha=0.9,
            )

            ax.axvline(
                x=x_r,
                color="k",
                lw=0.8,
                ls="--",
                alpha=0.25,
            )

    ax.legend()
    ax.set_title(title,pad=30)
    plt.tight_layout()

    if savepath is not None:
        outdir = Path(savepath)
        outdir.mkdir(parents=True, exist_ok=True)
        outfile = outdir / title_to_filename(title)
        plt.savefig(outfile, dpi=300, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close(fig)



def plot_fig9B(batch_records,title,show=True,savepath=None):

    """
    Plot intrinsic dimension of the last hidden layer together with training/test
    accuracy across iterations (reproduction of Fig. 9B).

    Args:
        batch_records (list):
            Each item corresponds to one training mini-batch and has the form:
                [iteration, train_acc, test_acc, last_id]

    Notes:
        - ID values are plotted on the left y-axis.
        - Training and test accuracy are plotted on the right y-axis.
        - This function only visualizes precomputed quantities; it does not compute ID itself.
    """
    batch_records = np.array(batch_records, dtype=object)

    iters = batch_records[:,0].astype(int)
    train_acc = batch_records[:,1].astype(float)
    test_acc  = batch_records[:,2]
    id_vals   = batch_records[:,3]


    mask_id = np.array([v is not None for v in id_vals])
    id_iters = iters[mask_id]
    id_vals  = id_vals[mask_id].astype(float)


    mask_test = np.array([v is not None for v in test_acc])
    test_iters = iters[mask_test]
    test_acc   = test_acc[mask_test].astype(float)

    train_acc_pct = train_acc * 100.0
    test_acc_pct  = test_acc * 100.0

    fig, ax1 = plt.subplots(figsize=(8,6))

    ax1.plot(id_iters, id_vals, 'o-', color='black', label="ID last hidden layer")
    ax1.set_xlabel("iterations (n. of mini-batches)")
    ax1.set_ylabel("ID")
    ax1.set_ylim(0, max(id_vals)*1.2)

    ax2 = ax1.twinx()
    ax2.plot(iters, train_acc_pct, linestyle='--', color='red', alpha=0.6, label="training accuracy")
    ax2.plot(test_iters, test_acc_pct, linestyle='-', color='blue', label="test accuracy")
    ax2.set_ylabel("accuracy (%)")
    ax2.set_ylim(0, 100)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
    ax1.set_title(title)
    plt.tight_layout()

    if savepath is not None:
        outdir = Path(savepath)
        outdir.mkdir(parents=True, exist_ok=True)
        outfile = outdir / title_to_filename(title)
        plt.savefig(outfile, dpi=300, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close(fig)




def plot_fig9C(epoch_records,title,show=True,savepath=None):


    """
    Plot the relationship between test error and intrinsic dimension of the last hidden
    layer across epochs (reproducing Fig. 9C).

    Args:
        epoch_records (list):
            Each entry corresponds to one epoch and has the format:
                [epoch_idx, test_acc, id_list]

            - epoch_idx (int):
                Epoch number.

            - test_acc (float):
                Test accuracy in [0, 1]. Converted internally to test error (%).

            - id_list (list of floats):
                Intrinsic-dimension values for each layer at that epoch.
                The last hidden layer is taken as id_list[-2].

    Notes:
        - Points are colored by epoch number (Viridis colormap).
        - Highlighted epochs include: 0, 1, 2, 5, 10, and the final epoch.
        - This function only visualizes precomputed ID values and does not
          perform ID estimation itself.
    """

    epochs = []
    errors = []
    ids_last = []

    for rec in epoch_records:
        epoch = rec[0]
        acc   = rec[1]
        id_list = rec[2]


        err = (1.0 - float(acc)) * 100.0

        mean_id = id_list[-2]

        epochs.append(epoch)
        errors.append(err)
        ids_last.append(float(mean_id))

    epochs = np.array(epochs, dtype=float)
    errors = np.array(errors, dtype=float)
    ids_last = np.array(ids_last, dtype=float)

    fig, ax = plt.subplots(figsize=(6,6))

    cmap = cm.get_cmap("viridis")
    norm = Normalize(vmin=epochs.min(), vmax=epochs.max())

    sc = ax.scatter(errors, ids_last,
                     c=epochs, cmap=cmap, norm=norm,
                     s=20, edgecolors='none')

    ax.set_xlabel("error (%)")
    ax.set_ylabel("ID")

    ymin = max(0, ids_last.min() - 1)
    ymax = ids_last.max() + 1
    ax.set_ylim(ymin, ymax)

    cbar = plt.colorbar(sc)
    cbar.set_label("epoch")

    highlight_epochs = [0, 1, 2, 5, 10, len(epochs)-1]
    for ep in highlight_epochs:
        if ep in epochs:
            i = np.where(epochs == ep)[0][0]
            ax.scatter(errors[i], ids_last[i],
                        s=80, facecolors='none', edgecolors='lightgreen', linewidths=1.5)
            ax.text(errors[i]+1, ids_last[i],
                     f"EPOCH {int(ep)}",
                     fontsize=8, color="seagreen", rotation=40)


    ax.set_title(title)
    plt.tight_layout()

    if savepath is not None:
        outdir = Path(savepath)
        outdir.mkdir(parents=True, exist_ok=True)
        outfile = outdir / title_to_filename(title)
        plt.savefig(outfile, dpi=300, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close(fig)



def rescale_to_range(arr, new_min=0, new_max=400):
    arr = np.asarray(arr, dtype=float)
    mn, mx = arr.min(), arr.max()
    if mx == mn:
        return np.full_like(arr, new_min, dtype=float)
    return (arr - mn) / (mx - mn) * (new_max - new_min) + new_min


def plot_fig5c(model_name, list_epoch_0_and_epoch_last, depths,ed_rescale_max,
               title,annotate=False,show=True,savepath=None):


    """
    Plot Fig.5C-style curves:
    ID (trained/untrained), PC-ID (trained/untrained), and rescaled ED vs relative depth.

    list_epoch_0_and_epoch_last:
        [ record_epoch0, record_epoch_last ],
        where each record is:
        [epoch_idx, test_acc, ID_list, ED_list, PCA_dim_list]
    """

    d_tr = list_epoch_0_and_epoch_last[1]
    d_un = list_epoch_0_and_epoch_last[0]

    x = depths / depths.max()

    y_id_tr = np.asarray(d_tr[2], dtype=float)
    y_id_un = np.asarray(d_un[2], dtype=float)

    y_pcid_tr = np.asarray(d_tr[4], dtype=float)
    y_pcid_un = np.asarray(d_un[4], dtype=float)

    y_ed_rescaled = rescale_to_range(np.asarray(d_tr[3], dtype=float),
                                     new_min=0, new_max=ed_rescale_max)

    plt.figure(figsize=(8,5))

    # ID
    plt.plot(x, y_id_tr, "o-",  color="k",        lw=2, ms=6, label="ID trained")
    plt.plot(x, y_id_un, "o--", color="k",  alpha=0.8, lw=2, ms=6, label="ID untrained")

    # PC-ID
    plt.plot(x, y_pcid_tr, "o-",  color="#d62728",        lw=2, ms=6, label="PC-ID trained")
    plt.plot(x, y_pcid_un, "o--", color="#d62728", alpha=0.8, lw=2, ms=6, label="PC-ID untrained")

    # ED (rescaled)
    plt.plot(x, y_ed_rescaled, "-", color="#1f77b4", lw=2, label="ED (rescaled)")

    plt.xlabel("relative depth", fontsize=12)
    plt.ylabel("intrinsic dimension", fontsize=12)
    plt.xlim(-0.02, 1.02)
    plt.xticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])

    ymax = max(y_id_tr.max(), y_id_un.max(),
               y_pcid_tr.max(), y_pcid_un.max(),
               y_ed_rescaled.max())
    plt.ylim(0, np.ceil(ymax*1.1 / 10) * 10)

    h, l = plt.gca().get_legend_handles_labels()
    bylabel = dict(zip(l, h))
    plt.legend(bylabel.values(), bylabel.keys(),
               frameon=False, bbox_to_anchor=(1.02, 1),
               loc="upper left", fontsize=11)

    plt.title(title, fontsize=13, pad=30)

    plt.tight_layout()

    if annotate:
        depth_names = list_epoch_0_and_epoch_last[0][-1]
        x_rel = x
        ax = plt.gca()

        y_top = ax.get_ylim()[1]
        for i in range(len(depth_names) - 1):
            x_l = x_rel[i]
            x_r = x_rel[i + 1]
            x_mid = 0.5 * (x_l + x_r)

            label = depth_names[i + 1]

            ax.text(
                x_mid,
                y_top * 1.03,
                label,
                ha="center",
                va="bottom",
                fontsize=8,
                color="black",
                rotation=0,
            )

            ax.axvline(
                x=x_r,
                color="k",
                lw=0.8,
                ls="--",
                alpha=0.25,
            )
    if savepath is not None:
        outdir = Path(savepath)
        outdir.mkdir(parents=True, exist_ok=True)
        outfile = outdir / title_to_filename(title)
        plt.savefig(outfile, dpi=300, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close(plt.gcf())
